# Practical 3: Cleaning and Preprocessing the Structured Log Dataset

## Aim

To validate, clean and preprocess the structured security-log dataset while preserving the original values and documenting every transformation.

## Expected Outcome

After completing this practical, we will be able to:

- Convert timestamps into validated datetime values.
- Validate IP addresses and source ports.
- Distinguish null, empty and semantically missing values.
- Normalize text without destroying the original evidence.
- Separate mixed metadata into interpretable components.
- Detect exact duplicate content without automatically deleting repeated events.
- Add explicit data-quality flags.
- Produce a reproducible processed dataset.


## Principle

Raw evidence will be preserved. Cleaned and normalized values will be stored as derived fields so every transformation remains traceable.

In [1]:
from pathlib import Path
import sys

import pandas as pd

PROJECT_ROOT = (
    Path.cwd().parent
    if Path.cwd().name == "notebooks"
    else Path.cwd()
)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.ingestion.cj_parser import FIELD_NAMES

STRUCTURED_PATH = (
    PROJECT_ROOT
    / "data"
    / "interim"
    / "cj_structured.csv"
)

NULL_SENTINEL = r"\N"
CHUNK_SIZE = 100_000

assert STRUCTURED_PATH.exists(), (
    f"Structured dataset not found: {STRUCTURED_PATH}"
)

print("Dataset:", STRUCTURED_PATH)
print(
    "Size (MiB):",
    round(STRUCTURED_PATH.stat().st_size / (1024 ** 2), 2),
)
print("Raw fields:", list(FIELD_NAMES))
print("src exists:", (PROJECT_ROOT / "src").exists())

Dataset: c:\Users\diyas\Desktop\PDS-Log-IDS-Project\data\interim\cj_structured.csv
Size (MiB): 302.38
Raw fields: ['category_type', 'sub_key', 'timestamp', 'client_ip', 'source_port', 'user_agent', 'language', 'metadata']
src exists: True


In [2]:
from src.preprocessing import clean_structured_chunk

print("Import successful.")

Import successful.


## 1. Baseline Data-Quality Profile

We distinguish three states:

- **Null:** the value was absent in the raw record.
- **Empty string:** the field existed but contained no characters.
- **Whitespace-only:** the field contained only spaces or similar characters.

These states are measured before defining any cleaning rule.

In [3]:
field_columns = list(FIELD_NAMES)

null_counts = pd.Series(0, index=field_columns, dtype="int64")
empty_counts = pd.Series(0, index=field_columns, dtype="int64")
whitespace_counts = pd.Series(0, index=field_columns, dtype="int64")

total_rows = 0

for chunk in pd.read_csv(
    STRUCTURED_PATH,
    usecols=field_columns,
    dtype=str,
    keep_default_na=False,
    na_values=[NULL_SENTINEL],
    chunksize=CHUNK_SIZE,
):
    total_rows += len(chunk)

    null_counts += chunk.isna().sum()

    for column in field_columns:
        values = chunk[column]

        empty_counts[column] += values.eq("").sum()

        stripped = values.fillna("").astype(str).str.strip()
        whitespace_mask = (
            values.notna()
            & values.ne("")
            & stripped.eq("")
        )
        whitespace_counts[column] += whitespace_mask.sum()

quality_profile = pd.DataFrame({
    "null_count": null_counts,
    "empty_string_count": empty_counts,
    "whitespace_only_count": whitespace_counts,
})

quality_profile["null_percent"] = (
    quality_profile["null_count"] / total_rows * 100
)

quality_profile["empty_percent"] = (
    quality_profile["empty_string_count"] / total_rows * 100
)

print("Total rows:", f"{total_rows:,}")
display(quality_profile)

Total rows: 2,062,361


,null_count,empty_string_count,whitespace_only_count,null_percent,empty_percent
category_type,2035084,0,0,98.677390,0.000000
sub_key,2035084,11144,0,98.677390,0.540352
timestamp,0,0,0,0.000000,0.000000
client_ip,0,0,0,0.000000,0.000000
source_port,0,0,0,0.000000,0.000000
user_agent,26153,0,0,1.268110,0.000000
language,2000685,0,0,97.009447,0.000000
metadata,2014477,0,0,97.678195,0.000000


### Interpretation

The dataset contains structurally optional fields rather than uniformly complete
tabular attributes. Missing values will therefore be preserved instead of being
replaced using statistical imputation.

The `sub_key` field contains both null values and empty strings. These represent
different source states and must remain distinguishable.

No whitespace-only values were detected, so no whitespace-specific correction is
required.

In [4]:
from collections import Counter
from ipaddress import ip_address

ip_counts = Counter()

invalid_timestamp_count = 0
invalid_port_format_count = 0
non_integer_port_count = 0
out_of_range_port_count = 0

validation_columns = [
    "timestamp",
    "client_ip",
    "source_port",
]

for chunk in pd.read_csv(
    STRUCTURED_PATH,
    usecols=validation_columns,
    dtype=str,
    keep_default_na=False,
    na_values=[NULL_SENTINEL],
    chunksize=CHUNK_SIZE,
):
    parsed_timestamps = pd.to_datetime(
        chunk["timestamp"],
        errors="coerce",
        utc=True,
    )
    invalid_timestamp_count += parsed_timestamps.isna().sum()

    ports = pd.to_numeric(
        chunk["source_port"],
        errors="coerce",
    )

    invalid_port_format_count += ports.isna().sum()

    numeric_ports = ports.dropna()
    non_integer_port_count += numeric_ports.mod(1).ne(0).sum()
    out_of_range_port_count += (
        ~numeric_ports.between(0, 65535)
    ).sum()

    ip_counts.update(chunk["client_ip"].dropna())

invalid_ip_values = {}
ip_version_counts = Counter()

for value, occurrence_count in ip_counts.items():
    try:
        parsed_ip = ip_address(value)
        ip_version_counts[parsed_ip.version] += occurrence_count
    except ValueError:
        invalid_ip_values[value] = occurrence_count

print("Invalid timestamps:", invalid_timestamp_count)
print("Invalid port formats:", invalid_port_format_count)
print("Non-integer ports:", non_integer_port_count)
print("Out-of-range ports:", out_of_range_port_count)
print("Unique invalid IP values:", len(invalid_ip_values))
print(
    "Records with invalid IPs:",
    sum(invalid_ip_values.values()),
)
print("IP-version record counts:", dict(ip_version_counts))

Invalid timestamps: 0
Invalid port formats: 0
Non-integer ports: 0
Out-of-range ports: 0
Unique invalid IP values: 0
Records with invalid IPs: 0
IP-version record counts: {4: 2062361}


## 2. Timestamp Representation Audit

Timestamp syntax, timezone evidence, chronological range, and source ordering are
examined before normalization. A timezone will not be invented when the source
does not provide one.

In [5]:
from collections import Counter

timestamp_shape_counts = Counter()
timezone_counts = Counter()

minimum_time = None
maximum_time = None
previous_time = None
out_of_order_transitions = 0

for chunk in pd.read_csv(
    STRUCTURED_PATH,
    usecols=["timestamp"],
    dtype=str,
    keep_default_na=False,
    na_values=[NULL_SENTINEL],
    chunksize=CHUNK_SIZE,
):
    raw_time = chunk["timestamp"]

    shapes = raw_time.str.replace(
        r"\d",
        "D",
        regex=True,
    )
    timestamp_shape_counts.update(shapes.value_counts().to_dict())

    has_z_suffix = raw_time.str.contains(
        r"[zZ]$",
        regex=True,
        na=False,
    )
    has_numeric_offset = raw_time.str.contains(
        r"[+-]\d{2}:?\d{2}$",
        regex=True,
        na=False,
    )

    timezone_counts["z_suffix"] += int(has_z_suffix.sum())
    timezone_counts["numeric_offset"] += int(
        (has_numeric_offset & ~has_z_suffix).sum()
    )
    timezone_counts["no_timezone_marker"] += int(
        (~has_z_suffix & ~has_numeric_offset).sum()
    )

    comparable_time = pd.to_datetime(
        raw_time,
        errors="coerce",
        utc=True,
    )

    out_of_order_transitions += int(
        comparable_time.diff()
        .lt(pd.Timedelta(0))
        .sum()
    )

    if (
        previous_time is not None
        and comparable_time.iloc[0] < previous_time
    ):
        out_of_order_transitions += 1

    previous_time = comparable_time.iloc[-1]

    chunk_minimum = comparable_time.min()
    chunk_maximum = comparable_time.max()

    minimum_time = (
        chunk_minimum
        if minimum_time is None
        else min(minimum_time, chunk_minimum)
    )
    maximum_time = (
        chunk_maximum
        if maximum_time is None
        else max(maximum_time, chunk_maximum)
    )

print("Timestamp shapes:")
for shape, count in timestamp_shape_counts.most_common():
    print(f"  {shape}: {count:,}")

print("Timezone evidence:", dict(timezone_counts))
print("Minimum parsed time:", minimum_time)
print("Maximum parsed time:", maximum_time)
print("Out-of-order transitions:", out_of_order_transitions)

Timestamp shapes:
  DDDD-DD-DD DD:DD:DD: 2,062,361
Timezone evidence: {'z_suffix': 0, 'numeric_offset': 0, 'no_timezone_marker': 2062361}
Minimum parsed time: 2023-01-08 08:07:15+00:00
Maximum parsed time: 2024-02-19 21:44:01+00:00
Out-of-order transitions: 585


In [6]:
backward_jump_seconds = []
backward_jump_lines = []

previous_time = None

for chunk in pd.read_csv(
    STRUCTURED_PATH,
    usecols=["source_line", "timestamp"],
    dtype=str,
    keep_default_na=False,
    na_values=[NULL_SENTINEL],
    chunksize=CHUNK_SIZE,
):
    parsed_time = pd.to_datetime(
        chunk["timestamp"],
        errors="coerce",
    )

    source_lines = pd.to_numeric(
        chunk["source_line"],
        errors="raise",
    )

    if previous_time is not None:
        boundary_delta = (
            parsed_time.iloc[0] - previous_time
        ).total_seconds()

        if boundary_delta < 0:
            backward_jump_seconds.append(-boundary_delta)
            backward_jump_lines.append(
                int(source_lines.iloc[0])
            )

    time_differences = parsed_time.diff().dt.total_seconds()
    backward_mask = time_differences.lt(0)

    backward_jump_seconds.extend(
        (-time_differences[backward_mask]).tolist()
    )
    backward_jump_lines.extend(
        source_lines[backward_mask].astype(int).tolist()
    )

    previous_time = parsed_time.iloc[-1]

jump_summary = pd.Series(
    backward_jump_seconds,
    name="backward_jump_seconds",
)

display(
    jump_summary.describe(
        percentiles=[0.50, 0.90, 0.95, 0.99]
    ).to_frame()
)

print("Jumps greater than 1 minute:", (jump_summary > 60).sum())
print("Jumps greater than 1 hour:", (jump_summary > 3_600).sum())
print("Jumps greater than 1 day:", (jump_summary > 86_400).sum())

,backward_jump_seconds
count,585.000000
mean,1.167521
std,0.378314
min,1.000000
50%,1.000000
90%,2.000000
95%,2.000000
99%,2.000000
max,3.000000


Jumps greater than 1 minute: 0
Jumps greater than 1 hour: 0
Jumps greater than 1 day: 0


## 3. Conservative Cleaning Policy

Based on the quality audit:

1. No records will be deleted because every record satisfies the structural and
   semantic validation rules.
2. Missing optional fields will not be imputed.
3. Empty strings and null values will remain distinct.
4. Repeated `record_hash` values will be retained because separate `event_id`
   values represent separate observed occurrences.
5. Original fields will not be overwritten.
6. Derived canonical fields will be created for timestamp, IP address, IP version,
   and source port.
7. The timestamp timezone will be recorded as unknown because the source contains
   no timezone marker.
8. File order will be preserved despite the 585 small backward timestamp movements.
9. Text will not be lowercased or decoded because casing and encoding can contain
   cybersecurity evidence.

In [7]:
DERIVED_COLUMNS = [
    "timestamp_normalized",
    "client_ip_canonical",
    "ip_version",
    "source_port_normalized",
]

CLEANING_POLICY = {
    "delete_valid_rows": False,
    "impute_missing_values": False,
    "collapse_null_and_empty": False,
    "remove_repeated_records": False,
    "overwrite_raw_fields": False,
    "sort_output_by_timestamp": False,
    "timestamp_timezone": "unknown",
    "derived_columns": DERIVED_COLUMNS,
}

display(pd.Series(CLEANING_POLICY, name="policy"))

delete_valid_rows                                                       False
impute_missing_values                                                   False
collapse_null_and_empty                                                 False
remove_repeated_records                                                 False
overwrite_raw_fields                                                    False
sort_output_by_timestamp                                                False
timestamp_timezone                                                    unknown
derived_columns             [timestamp_normalized, client_ip_canonical, ip...
Name: policy, dtype: object

In [8]:
from src.preprocessing import (
    clean_structured_chunk,
    DERIVED_COLUMNS,
)
test_chunk = pd.read_csv(
    STRUCTURED_PATH,
    nrows=5,
    dtype=str,
    keep_default_na=False,
    na_values=[NULL_SENTINEL],
)

cleaned_test = clean_structured_chunk(test_chunk)

pd.testing.assert_frame_equal(
    test_chunk,
    cleaned_test[list(test_chunk.columns)],
)

assert cleaned_test["client_ip_canonical"].notna().all()
assert cleaned_test["ip_version"].isin([4, 6]).all()
assert cleaned_test["source_port_normalized"].between(
    0, 65535
).all()

print("Original fields unchanged: passed")
print("Derived-field validation: passed")

display(
    cleaned_test[
        [
            "timestamp",
            "timestamp_normalized",
            "source_port",
            "source_port_normalized",
            "ip_version",
        ]
    ]
)

Original fields unchanged: passed
Derived-field validation: passed


,timestamp,timestamp_normalized,source_port,source_port_normalized,ip_version
0,2023-01-08 08:07:15,2023-01-08 08:07:15,61901,61901,4
1,2023-01-08 08:07:16,2023-01-08 08:07:16,22667,22667,4
2,2023-01-08 08:07:25,2023-01-08 08:07:25,62901,62901,4
3,2023-01-08 08:07:26,2023-01-08 08:07:26,61879,61879,4
4,2023-01-08 08:07:34,2023-01-08 08:07:34,46994,46994,4


## 4. Full-Dataset Cleaning

The evidence-preserving transformation is applied in chunks. Output is first
written to temporary files and published only after row-count and integrity
checks pass.

In [9]:
import json
from datetime import datetime, timezone

from src.preprocessing import clean_structured_chunk
from src.provenance import calculate_file_sha256


CLEANED_OUTPUT = (
    PROJECT_ROOT / "data" / "processed" / "cj_cleaned.csv"
)

CLEANING_MANIFEST_OUTPUT = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "cj_cleaning_manifest.json"
)

CLEANED_OUTPUT.parent.mkdir(parents=True, exist_ok=True)


def export_cleaned_dataset():
    output_partial = CLEANED_OUTPUT.with_name(
        CLEANED_OUTPUT.name + ".partial"
    )
    manifest_partial = CLEANING_MANIFEST_OUTPUT.with_name(
        CLEANING_MANIFEST_OUTPUT.name + ".partial"
    )

    if CLEANED_OUTPUT.exists() or CLEANING_MANIFEST_OUTPUT.exists():
        raise FileExistsError(
            "Final cleaning outputs already exist. "
            "Verify them instead of silently overwriting them."
        )

    for partial_path in [output_partial, manifest_partial]:
        if partial_path.exists():
            partial_path.unlink()

    input_rows = 0
    output_rows = 0
    chunk_count = 0
    write_header = True

    try:
        for chunk in pd.read_csv(
            STRUCTURED_PATH,
            dtype=str,
            keep_default_na=False,
            na_values=[NULL_SENTINEL],
            chunksize=CHUNK_SIZE,
        ):
            cleaned_chunk = clean_structured_chunk(chunk)

            if not chunk.equals(
                cleaned_chunk[list(chunk.columns)]
            ):
                raise RuntimeError(
                    "A raw source field changed during cleaning."
                )

            cleaned_chunk.to_csv(
                output_partial,
                mode="w" if write_header else "a",
                header=write_header,
                index=False,
                na_rep=NULL_SENTINEL,
            )

            input_rows += len(chunk)
            output_rows += len(cleaned_chunk)
            chunk_count += 1
            write_header = False

        if input_rows != output_rows:
            raise RuntimeError(
                "Input and output row counts do not match."
            )

        output_sha256 = calculate_file_sha256(output_partial)

        manifest = {
    "created_at_utc": datetime.now(
        timezone.utc
    ).isoformat(),
    "input_file": (
        STRUCTURED_PATH
        .relative_to(PROJECT_ROOT)
        .as_posix()
    ),
    "output_file": (
        CLEANED_OUTPUT
        .relative_to(PROJECT_ROOT)
        .as_posix()
    ),
    "input_sha256": calculate_file_sha256(
        STRUCTURED_PATH
    ),
    "output_sha256": output_sha256,
    "input_rows": input_rows,
    "output_rows": output_rows,
    "chunk_count": chunk_count,
    "chunk_size": CHUNK_SIZE,
    "null_sentinel": NULL_SENTINEL,
    "timestamp_timezone": "unknown",
    "timestamp_format": "%Y-%m-%d %H:%M:%S",
    "derived_columns": list(DERIVED_COLUMNS),
    "cleaning_policy": CLEANING_POLICY,
}

        with manifest_partial.open(
            "w",
            encoding="utf-8",
        ) as file:
            json.dump(manifest, file, indent=2)

        output_partial.replace(CLEANED_OUTPUT)
        manifest_partial.replace(CLEANING_MANIFEST_OUTPUT)

        return manifest

    except Exception:
        for partial_path in [output_partial, manifest_partial]:
            if partial_path.exists():
                partial_path.unlink()
        raise

if (
    CLEANED_OUTPUT.exists()
    and CLEANING_MANIFEST_OUTPUT.exists()
):
    with CLEANING_MANIFEST_OUTPUT.open(
        "r",
        encoding="utf-8",
    ) as file:
        cleaning_result = json.load(file)

    print("Reusing existing cleaned dataset.")

elif (
    CLEANED_OUTPUT.exists()
    or CLEANING_MANIFEST_OUTPUT.exists()
):
    raise RuntimeError(
        "Only one cleaning output exists. "
        "Inspect the incomplete output before continuing."
    )

else:
    cleaning_result = export_cleaned_dataset()

cleaning_result

Reusing existing cleaned dataset.


{'created_at_utc': '2026-08-18T05:39:31.088494+00:00',
 'input_file': 'data/interim/cj_structured.csv',
 'output_file': 'data/processed/cj_cleaned.csv',
 'input_sha256': 'a149f06bcced34426dcc02812366e010caa124d1e901f2b640467238a0dacc6a',
 'output_sha256': 'c022cee1cb2de32f7bb1db1d38d5d1dad1079b2fb590f854f9aead7fa54995f2',
 'input_rows': 2062361,
 'output_rows': 2062361,
 'chunk_count': 21,
 'chunk_size': 100000,
 'null_sentinel': '\\N',
 'timestamp_timezone': 'unknown',
 'timestamp_format': '%Y-%m-%d %H:%M:%S',
 'derived_columns': ['timestamp_normalized',
  'client_ip_canonical',
  'ip_version',
  'source_port_normalized'],
 'cleaning_policy': {'delete_valid_rows': False,
  'impute_missing_values': False,
  'collapse_null_and_empty': False,
  'remove_repeated_records': False,
  'overwrite_raw_fields': False,
  'sort_output_by_timestamp': False,
  'timestamp_timezone': 'unknown',
  'derived_columns': ['timestamp_normalized',
   'client_ip_canonical',
   'ip_version',
   'source_port_nor

In [10]:
with CLEANING_MANIFEST_OUTPUT.open(
    "r",
    encoding="utf-8",
) as file:
    cleaning_manifest = json.load(file)

verified_rows = 0
derived_null_counts = pd.Series(
    0,
    index=DERIVED_COLUMNS,
    dtype="int64",
)

input_reader = pd.read_csv(
    STRUCTURED_PATH,
    dtype=str,
    keep_default_na=False,
    na_values=[NULL_SENTINEL],
    chunksize=CHUNK_SIZE,
)

output_reader = pd.read_csv(
    CLEANED_OUTPUT,
    dtype=str,
    keep_default_na=False,
    na_values=[NULL_SENTINEL],
    chunksize=CHUNK_SIZE,
)

for input_chunk, output_chunk in zip(
    input_reader,
    output_reader,
    strict=True,
):
    expected_columns = [
        *input_chunk.columns,
        *DERIVED_COLUMNS,
    ]

    if list(output_chunk.columns) != expected_columns:
        raise AssertionError(
            "Cleaned output schema is incorrect."
        )

    pd.testing.assert_frame_equal(
        input_chunk,
        output_chunk[list(input_chunk.columns)],
    )

    derived_null_counts += (
        output_chunk[list(DERIVED_COLUMNS)]
        .isna()
        .sum()
    )

    verified_rows += len(output_chunk)

actual_output_hash = calculate_file_sha256(
    CLEANED_OUTPUT
)

partial_files = list(
    CLEANED_OUTPUT.parent.glob("*.partial")
)

assert verified_rows == 2_062_361
assert verified_rows == cleaning_manifest["output_rows"]
assert derived_null_counts.sum() == 0
assert actual_output_hash == cleaning_manifest["output_sha256"]
assert not partial_files

print("Verified rows:", f"{verified_rows:,}")
print("Raw-field round trip: passed")
print("Derived fields complete: passed")
print("Output fingerprint: passed")
print("Partial files remaining:", len(partial_files))
print(
    "Cleaned size (MiB):",
    round(CLEANED_OUTPUT.stat().st_size / (1024 ** 2), 2),
)

Verified rows: 2,062,361
Raw-field round trip: passed
Derived fields complete: passed
Output fingerprint: passed
Partial files remaining: 0
Cleaned size (MiB): 386.73


## Findings and Conclusion

- All 2,062,361 structured records were processed.
- No records were deleted or imputed.
- Null values and empty strings remained distinguishable.
- All timestamps, client IP addresses, and source ports passed semantic validation.
- All observed client addresses were IPv4.
- Timestamps used one consistent second-level format but contained no timezone
  evidence, so the timezone remains recorded as unknown.
- File order contained 585 minor backward movements of only 1–3 seconds.
- Original file order and provenance fields were preserved.
- Four derived canonical fields were added without overwriting raw evidence.
- The final dataset passed full round-trip and SHA-256 integrity verification.
- The cleaning process is chunked, atomic, evidence-preserving, and idempotent.

Cleaning does not necessarily mean modifying data. In forensic datasets, validation
and justified preservation can be more correct than aggressive correction.